# ___MR-PMM___
----------------------

In [110]:
set.seed(20206 - 3 - 9)
library("ape")
library("phytools")

In [111]:
# multi response phylogenetic mixed effect models
# https://benjamin-halliwell.github.io/MR-PMM/MR-PMM_euc_example_analysis.html

In [8]:
data <- read.csv("../../data/chapter2/FREDv3subset/collab_fineroots_log_995_species_means_5states_name_matched_with_phylogeny.csv") # log transformed SRL and RD species averages
phylogeny <- ape::read.tree("../../data/chapter2/uphylomaker/collab_fineroots_log_995_species_means_5states.tre") # phylogenetic tree
stopifnot(data$binominal==phylogeny$tip.label)

In [9]:
ape::is.binary(phylogeny) # damn

[1] FALSE

In [11]:
ape::is.ultrametric(phylogeny) # :)

[1] TRUE

In [15]:
# make the phylogeny completely bifurcating by introducing 0 length branches
phylogeny <- ape::multi2di(phylogeny)
sum(phylogeny$edge.length == 0) # damn

[1] 91

In [21]:
phylogeny$edge.length[phylogeny$edge.length==0] <- rnorm(n = sum(phylogeny$edge.length == 0), mean = 1e-6, sd = 1e-8) # replace the 0 length edges with random noise
sum(phylogeny$edge.length == 0) # okay

[1] 0

In [ ]:
# "By including more diverse species in our phylogeny, we capture deeper splits that represent more meaningful divergences in the genotype and phenotype of extant lineages,
# precisely the effects we intend to model when analysing inter-species data."
# "One consequence of this, is that shallow topology (near the tips) is less informative than deep topology when attempting to infer patterns of phylogenetic niche conservatism, because differences between genera
# are usually more significant than differences between species within genera."

In [ ]:
# "For higher taxonomic ranks (e.g. genus), it will usually be possible to derive a unique consensus tree by sampling a single species from each genus and simply pruning the tree to those tips.
# This approach may be problematic for lower taxonomic ranks however, because more closely related species are less likely to be monophyletic with respect to the taxonomic rank in question.
# Even in such cases, we can easily account for this phylogenetic uncertainty by randomly sampling topologies at the specified rank and fitting our models over this sample of trees."

In [108]:
# transform our species level phylogeny to genus level phylogeny

NITERATIONS = 100
sampled_subphylogenies <- list()

for (i in 1:NITERATIONS) {
    sampled_species <- unname(mapply(split.data.frame(data[, c("binominal", "F01286")], ~F01286), FUN = function(df) sample(df$binominal, 1))) # this is a vector of one randomly sampled species per every genera in the phylogeny
    subphylo <- ape::keep.tip(phy = phylogeny, tip = sampled_species) # trimmed phylogeny with one randomly sampled species per genus
    if(!ape::is.binary(subphylo)) subphylo <- ape::multi2di(subphylo) # make bifucracting if not already
    if(!ape::is.ultrametric(subphylo)) subphylo <- phytools::force.ultrametric(subphylo, method = "extend", message=FALSE) # make ultrametric if not already


    ape::vcv.phylo(phy = subphylo, corr = TRUE)
}



Phylogenetic tree with 516 tips and 515 internal nodes.

Tip labels:
  Rudbeckia_hirta, Ratibida_pinnata, Heliopsis_helianthoides, Liatris_aspera, Arnica_fulgens, Hymenoxys_richardsonii, ...
Node labels:
  , , Spermatophyta, Mesangiospermae, mrcaott2ott121, eudicotyledons, ...

Rooted; includes branch length(s).